## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/mlflow_research'
os.makedirs(PROJECT, exist_ok=True)
print("Project folder:", PROJECT)
print("Contents:", sorted(os.listdir(PROJECT)))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/mlflow_research
Contents: ['candidates.csv', 'candidates_licensed.csv', 'detector_processed.txt', 'mlflow_files.csv', 'mlflow_files.gsheet', 'mlflow_repos.csv', 'mlflow_repos_widened.csv', 'prefilter_processed.txt', 'results.csv.gz', 'widened_processed.txt']


## 2. Setup: GitHub token

In [ ]:
from getpass import getpass
GITHUB_TOKEN = getpass("Paste the token, then press Enter: ")
print("Token captured, length:", len(GITHUB_TOKEN))

Paste the token Corey sent, then press Enter: ··········
Token captured, length: 93


## 3. Widened pre-filter

In [ ]:
import base64, time, os, requests, pandas as pd

MANIFEST_NAMES = {
    "requirements.txt","requirements-dev.txt","pyproject.toml","setup.py",
    "setup.cfg","environment.yml","environment.yaml","Pipfile","conda.yaml",
}
MAX_MANIFESTS_PER_REPO = 25   # safety cap for monorepos with hundreds of manifests

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})

def gh_get(url):
    while True:
        r = session.get(url)
        if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
            wait = max(int(r.headers.get("X-RateLimit-Reset", time.time()+60)) - int(time.time()) + 1, 1)
            print(f"  rate limit, sleeping {wait}s"); time.sleep(wait); continue
        return r

def default_branch(full):
    r = gh_get(f"https://api.github.com/repos/{full}")
    return r.json().get("default_branch") if r.status_code == 200 else None

def all_manifest_paths(full):
    br = default_branch(full)
    if not br: return None
    r = gh_get(f"https://api.github.com/repos/{full}/git/trees/{br}?recursive=1")
    if r.status_code != 200: return None
    tree = r.json().get("tree", [])
    paths = [t["path"] for t in tree
             if t.get("type") == "blob" and t["path"].split("/")[-1] in MANIFEST_NAMES]
    return paths[:MAX_MANIFESTS_PER_REPO]

def mentions_mlflow(full, path):
    r = gh_get(f"https://api.github.com/repos/{full}/contents/{path}")
    if r.status_code != 200: return False
    d = r.json()
    if d.get("encoding") != "base64": return False
    try: return "mlflow" in base64.b64decode(d["content"]).decode("utf-8","ignore").lower()
    except Exception: return False

# --- work list: all licensed+dated candidates MINUS repos already kept by v2 ---
all_repos = pd.read_csv(f"{PROJECT}/candidates.csv")["name"].dropna().tolist()
kept_v2 = set(pd.read_csv(f"{PROJECT}/mlflow_repos.csv")["repo"].dropna().tolist())

PROC = f"{PROJECT}/widened_processed.txt"
OUT  = f"{PROJECT}/mlflow_repos_widened.csv"

processed = set()
if os.path.exists(PROC):
    processed = {l.strip() for l in open(PROC) if l.strip()}
if not os.path.exists(OUT):
    open(OUT, "w").write("repo,evidence_file,evidence_path\n")

todo = [r for r in all_repos if r not in kept_v2 and r not in processed]
kept_new = sum(1 for _ in open(OUT)) - 1
print(f"Candidates {len(all_repos)} | already kept by v2 {len(kept_v2)} | done this pass {len(processed)} | remaining {len(todo)} | new keeps so far {kept_new}\n")

out_f = open(OUT, "a"); proc_f = open(PROC, "a")
for i, full in enumerate(todo, 1):
    paths = all_manifest_paths(full)
    if paths:
        for p in paths:
            if mentions_mlflow(full, p):
                fname = p.split("/")[-1]
                out_f.write(f"{full},{fname},{p}\n"); out_f.flush(); kept_new += 1
                print(f"KEEP {full} ({p})  new keeps: {kept_new}"); break
    proc_f.write(full + "\n"); proc_f.flush()
    if i % 200 == 0: print(f"[{i}/{len(todo)}] new keeps: {kept_new}")
    time.sleep(0.03)
out_f.close(); proc_f.close()
print(f"\nFinished. New MLflow repos found by widened scan: {kept_new}")

Candidates 9154 | already kept by v2 52 | done this pass 2006 | remaining 7096 | new keeps so far 7

  rate limit, sleeping 3284s
[200/7096] new keeps: 7
[400/7096] new keeps: 7
[600/7096] new keeps: 7
  rate limit, sleeping 3100s
KEEP Kilo-Org/kilo-marketplace (skills/azureml-scaffolding/assets/src/mypkg/pyproject.toml)  new keeps: 8
[800/7096] new keeps: 8
KEEP aws-samples/sample-bedrock-migration-and-modernization-tools (usecase-examples/financial-compliance-agent-eval/haystack-intro/requirements.txt)  new keeps: 9
[1000/7096] new keeps: 9
  rate limit, sleeping 3136s
[1200/7096] new keeps: 9
[1400/7096] new keeps: 9
  rate limit, sleeping 3146s
[1600/7096] new keeps: 9
[1800/7096] new keeps: 9
KEEP microsoft/agent-governance-toolkit (agent-governance-python/agent-governance-toolkit-cli/pyproject.toml)  new keeps: 10
[2000/7096] new keeps: 10
  rate limit, sleeping 3082s


## 4. Combine v2 + widened keeps
Produces `mlflow_repos_combined.csv`, the input for the detector.

In [ ]:
import pandas as pd, os

v2 = pd.read_csv(f"{PROJECT}/mlflow_repos.csv")[["repo","evidence_file"]]
v2["evidence_path"] = v2["evidence_file"]   # v2 evidence was always at repo root
wide = pd.read_csv(f"{PROJECT}/mlflow_repos_widened.csv")

combined = pd.concat([v2, wide], ignore_index=True).drop_duplicates(subset=["repo"])
combined.to_csv(f"{PROJECT}/mlflow_repos_combined.csv", index=False)
print(f"v2 keeps: {len(v2)} | widened new keeps: {len(wide)} | combined unique repos: {len(combined)}")

## 5. Detector over NEW repos only
Same AST detector as v2. Reads the combined list but skips repos already processed by the
v2 detector, so only newly found repos are scanned. Appends to the same `mlflow_files.csv`.

In [ ]:
import ast, base64, time, requests, csv, os
import pandas as pd

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"})

def gh_get(url):
    while True:
        r = session.get(url)
        if r.status_code == 403 and r.headers.get("X-RateLimit-Remaining") == "0":
            wait = max(int(r.headers.get("X-RateLimit-Reset", time.time()+60)) - int(time.time()) + 1, 1)
            print(f"  rate limit, sleeping {wait}s"); time.sleep(wait); continue
        return r

def code_search(full):
    paths, page = [], 1
    while True:
        r = gh_get(f"https://api.github.com/search/code?q=mlflow+repo:{full}+language:python&per_page=100&page={page}")
        if r.status_code != 200:
            print(f"  search failed {full}: {r.status_code}"); break
        items = r.json().get("items", [])
        paths += [it["path"] for it in items]
        if len(items) < 100: break
        page += 1; time.sleep(7)
    return paths

def fetch(full, path):
    r = gh_get(f"https://api.github.com/repos/{full}/contents/{path}")
    if r.status_code != 200: return None
    d = r.json()
    if d.get("encoding") != "base64": return None
    try: return base64.b64decode(d["content"]).decode("utf-8","ignore")
    except Exception: return None

def analyze(src):
    has_import, n_calls = False, 0
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return ("import mlflow" in src or "from mlflow" in src), src.count("mlflow.")
    for node in ast.walk(tree):
        if isinstance(node, ast.Import) and any(n.name=="mlflow" or n.name.startswith("mlflow.") for n in node.names):
            has_import = True
        elif isinstance(node, ast.ImportFrom) and node.module and (node.module=="mlflow" or node.module.startswith("mlflow.")):
            has_import = True
        elif isinstance(node, ast.Call):
            f = node.func
            while isinstance(f, ast.Attribute):
                if isinstance(f.value, ast.Name) and f.value.id=="mlflow":
                    n_calls += 1; break
                f = f.value
    return has_import, n_calls

repos = pd.read_csv(f"{PROJECT}/mlflow_repos_combined.csv")["repo"].dropna().tolist()

PROC_OLD = f"{PROJECT}/detector_processed.txt"        # v2 detector checkpoint
PROC_NEW = f"{PROJECT}/detector_processed_v3.txt"
OUT      = f"{PROJECT}/mlflow_files.csv"              # append to the SAME dataset file

done = set()
for pth in (PROC_OLD, PROC_NEW):
    if os.path.exists(pth):
        done |= {l.strip() for l in open(pth) if l.strip()}
if not os.path.exists(OUT):
    with open(OUT, "w", newline="") as f:
        csv.writer(f).writerow(["repo","file_path","has_import","n_calls"])

todo = [r for r in repos if r not in done]
print(f"Combined repos {len(repos)} | already detected {len(done & set(repos))} | remaining {len(todo)}\n")

out_f = open(OUT, "a", newline=""); w = csv.writer(out_f)
proc_f = open(PROC_NEW, "a")
total_files = 0
for full in todo:
    print(f"\n{full}")
    paths = code_search(full)
    print(f"  {len(paths)} candidate file(s)")
    for p in paths:
        src = fetch(full, p)
        if not src: continue
        imp, calls = analyze(src)
        if imp or calls:
            w.writerow([full, p, imp, calls]); out_f.flush(); total_files += 1
            print(f"  MLflow file: {p}  (import={imp}, calls={calls})")
    proc_f.write(full + "\n"); proc_f.flush()
    time.sleep(7)
out_f.close(); proc_f.close()
print(f"\nDone. {total_files} new MLflow files recorded.")

## 6. Dedupe + updated summary counts

In [ ]:
import pandas as pd, os

f = pd.read_csv(f"{PROJECT}/mlflow_files.csv")
dups = f.duplicated().sum()
if dups:
    f = f.drop_duplicates()
    f.to_csv(f"{PROJECT}/mlflow_files.csv", index=False)
print(f"duplicates removed: {dups}")

raw = pd.read_csv(f"{PROJECT}/results.csv.gz")
lic = pd.read_csv(f"{PROJECT}/candidates_licensed.csv")
comb = pd.read_csv(f"{PROJECT}/mlflow_repos_combined.csv")
print(f"SEART export:                 {len(raw)}")
print(f"After license filter:         {len(lic)}")
print(f"After widened MLflow filter:  {len(comb)}")
print(f"Repos with MLflow files:      {f['repo'].nunique()}")
print(f"Total MLflow files:           {len(f)}")

## 7. New 10-sample validation (rerun after the widened set is final)


In [ ]:
import pandas as pd, random

files_df = pd.read_csv(f"{PROJECT}/mlflow_files.csv")
final_repos = sorted(files_df["repo"].unique())
print(f"Final repos with MLflow files: {len(final_repos)}\n")

random.seed(43)
sample = random.sample(final_repos, min(10, len(final_repos)))
for r in sample:
    rows = files_df[files_df["repo"] == r]
    print(f"=== {r}  ->  https://github.com/{r}")
    for _, row in rows.iterrows():
        print(f"    {row['file_path']}  (import={row['has_import']}, calls={row['n_calls']})")
    print()